# Демонстрация

В данном ноутбуке представлен пример кода для запуска демонстрационного пользовательского интерфейса библиотеки `manuscript-ocr` в версии `0.1.12` и выше.

Интерфейс демонстрирует работу OCR-пайплайна библиотеки, включая детекцию текста, распознавание и постобработку результата, и предназначен для интерактивной апробации возможностей проекта.

Реализация демонстрационного интерфейса в данном примере выполнена с использованием библиотеки `Gradio`.

`Gradio` не является частью библиотеки `manuscript-ocr`, не распространяется вместе с ней и не входит в обязательный набор зависимостей проекта, так как требуется только для запуска пользовательского интерфейса демонстрационной программы.

В отличие от интеграционных примеров с внешними компонентами, в данном ноутбуке используются только нейросетевые модели, реализованные в библиотеке `manuscript-ocr`: детектор `EAST`, распознаватель `TRBA` и корректор `CharLM`.

In [1]:
%pip install "gradio==5.50.0"

  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached brotli-1.2.0-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.135.3-py3-none-any.whl.metadata (28 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
  Using cached orjson-3.11.8-cp311-cp311-win_amd64.whl.metadata (43 kB)
  Using cached pillow-11.3.0-cp311-cp311-win_amd64.whl.metadata (9.2 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.wh

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
manuscript-ocr 0.1.12 requires onnxruntime>=1.16, which is not installed.


In [ ]:
!pip install "manuscript-ocr>=0.1.12"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.2/574.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 106.2 MB/s eta 0:00:00


In [ ]:
import time
import tempfile

import gradio as gr
import numpy as np
from PIL import Image

from manuscript import CharLM, Pipeline
from manuscript.detectors import EAST, YOLO
from manuscript.recognizers import PPOCRv5Rec, TRBA


DETECTOR_MODELS = ["yolo26x_obb_text_g1", "yolo26s_obb_text_g1", "east_50_g1"]
CORRECTOR_MODELS = ["prereform_charlm_g1", "modern_charlm_g1"]
RECOGNIZER_MODELS = ["trba_lite_g2", "trba_lite_g1", "trba_base_g1", "custom_ppocrv5_rec_g2"]

DETECTOR_DEFAULTS = {
    "yolo26x_obb_text_g1": {
        "target_size": 1408,
        "score_thresh": 0.1,
        "expand_ratio_w": 1.4,
        "expand_ratio_h": 1.5,
        "is_east": False,
    },
    "yolo26s_obb_text_g1": {
        "target_size": 1408,
        "score_thresh": 0.1,
        "expand_ratio_w": 1.4,
        "expand_ratio_h": 1.5,
        "is_east": False,
    },
    "east_50_g1": {
        "target_size": 1280,
        "score_thresh": 0.6,
        "expand_ratio_w": 1.4,
        "expand_ratio_h": 1.5,
        "is_east": True,
    },
}

last_recognition_page = None
last_correction_page = None


def create_pipeline(
    detector_model,
    recognizer_model,
    corrector_model,
    target_size,
    score_thresh,
    expand_ratio_w,
    expand_ratio_h,
    mask_threshold,
    apply_threshold,
    max_edits,
):
    if detector_model.startswith("east_"):
        detector = EAST(
            weights=detector_model,
            target_size=int(target_size),
            score_thresh=score_thresh,
            expand_ratio_w=expand_ratio_w,
            expand_ratio_h=expand_ratio_h,
        )
    else:
        detector = YOLO(
            weights=detector_model,
            target_size=int(target_size),
            score_thresh=score_thresh,
        )

    if recognizer_model == "custom_ppocrv5_rec_g2":
        recognizer = PPOCRv5Rec(weights=recognizer_model)
    else:
        recognizer = TRBA(weights=recognizer_model)
    corrector = CharLM(
        weights=corrector_model,
        mask_threshold=mask_threshold,
        apply_threshold=apply_threshold,
        max_edits=int(max_edits),
    )

    return Pipeline(
        detector=detector,
        recognizer=recognizer,
        corrector=corrector,
    )


def update_detector_controls(detector_model):
    cfg = DETECTOR_DEFAULTS[detector_model]
    is_east = cfg["is_east"]

    return (
        gr.update(
            value=cfg["target_size"],
            interactive=False if not is_east else True,
            label="Размер изображения" if is_east else "Размер изображения (по модели)",
        ),
        gr.update(value=cfg["score_thresh"]),
        gr.update(
            value=cfg["expand_ratio_w"],
            visible=is_east,
            interactive=is_east,
        ),
        gr.update(
            value=cfg["expand_ratio_h"],
            visible=is_east,
            interactive=is_east,
        ),
    )


def count_words_in_page(page):
    if page is None:
        return 0

    count = 0
    for block in page.blocks:
        for line in block.lines:
            count += sum(1 for span in line.text_spans if span.text)
    return count


def highlight_differences(original, corrected):
    html = []
    i, j = 0, 0

    while i < len(original) or j < len(corrected):
        if i < len(original) and j < len(corrected):
            if original[i] == corrected[j]:
                if original[i] == "\n":
                    html.append("<br>")
                else:
                    html.append(corrected[j])
                i += 1
                j += 1
            else:
                html.append(
                    f'<span style="background-color: #90EE90; font-weight: bold;">{corrected[j]}</span>'
                )
                i += 1
                j += 1
        elif i < len(original):
            i += 1
        else:
            html.append(
                f'<span style="background-color: #90EE90; font-weight: bold;">{corrected[j]}</span>'
            )
            j += 1

    return f'<div style="white-space: pre-wrap; font-family: monospace;">{"".join(html)}</div>'


def process_image(
    image,
    detector_model,
    recognizer_model,
    corrector_model,
    target_size,
    score_thresh,
    expand_ratio_w,
    expand_ratio_h,
    mask_threshold,
    apply_threshold,
    max_edits,
):
    global last_recognition_page, last_correction_page

    if image is None:
        return None, "", "", ""

    try:
        pipeline = create_pipeline(
            detector_model,
            recognizer_model,
            corrector_model,
            target_size,
            score_thresh,
            expand_ratio_w,
            expand_ratio_h,
            mask_threshold,
            apply_threshold,
            max_edits,
        )

        start_time = time.time()
        _, vis_image = pipeline.predict(image, vis=True)
        elapsed_time = time.time() - start_time

        last_recognition_page = pipeline.last_recognition_page
        last_correction_page = pipeline.last_correction_page

        text_before = pipeline.get_text(last_recognition_page) if last_recognition_page else ""
        text_after = pipeline.get_text(last_correction_page) if last_correction_page else ""

        word_count = count_words_in_page(last_correction_page)
        pages_per_sec = 1.0 / elapsed_time if elapsed_time > 0 else 0.0
        words_per_sec = word_count / elapsed_time if elapsed_time > 0 else 0.0

        stats_text = (
            f"Время: {elapsed_time:.2f} сек | "
            f"{pages_per_sec:.2f} стр/сек | "
            f"{words_per_sec:.1f} слов/сек"
        )

        if isinstance(vis_image, np.ndarray):
            vis_image = Image.fromarray(vis_image)

        highlighted = highlight_differences(text_before, text_after)

        return vis_image, text_before, highlighted, stats_text

    except Exception as e:
        error_msg = f"Ошибка: {e}"
        return None, error_msg, "", error_msg


def save_recognition_json():
    global last_recognition_page

    if last_recognition_page is None:
        return None

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False,
        encoding="utf-8",
    ) as f:
        f.write(last_recognition_page.to_json())
        return f.name


def save_correction_json():
    global last_correction_page

    if last_correction_page is None:
        return None

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False,
        encoding="utf-8",
    ) as f:
        f.write(last_correction_page.to_json())
        return f.name


default_detector = "yolo26x_obb_text_g1"
default_cfg = DETECTOR_DEFAULTS[default_detector]

with gr.Blocks(title="OCR Pipeline", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Manuscript Demo")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Изображение", type="pil")

            with gr.Row():
                detector_selector = gr.Dropdown(
                    choices=DETECTOR_MODELS,
                    value=default_detector,
                    label="Детектор",
                )
                recognizer_selector = gr.Dropdown(
                    choices=RECOGNIZER_MODELS,
                    value="custom_ppocrv5_rec_g2",
                    label="Распознаватель",
                )
                corrector_selector = gr.Dropdown(
                    choices=CORRECTOR_MODELS,
                    value="prereform_charlm_g1",
                    label="Корректор",
                )

            with gr.Accordion("Параметры детектора", open=False):
                target_size = gr.Slider(
                    640,
                    2560,
                    value=default_cfg["target_size"],
                    step=64,
                    label="Размер изображения (по модели)",
                    interactive=False,
                )
                score_thresh = gr.Slider(
                    0.1,
                    0.9,
                    value=default_cfg["score_thresh"],
                    step=0.05,
                    label="Порог уверенности",
                )
                expand_ratio_w = gr.Slider(
                    0.5,
                    3.0,
                    value=default_cfg["expand_ratio_w"],
                    step=0.1,
                    label="Расширение по ширине (EAST)",
                    visible=False,
                    interactive=False,
                )
                expand_ratio_h = gr.Slider(
                    0.5,
                    3.0,
                    value=default_cfg["expand_ratio_h"],
                    step=0.1,
                    label="Расширение по высоте (EAST)",
                    visible=False,
                    interactive=False,
                )

            with gr.Accordion("Параметры корректора", open=False):
                mask_threshold = gr.Slider(
                    0.0, 0.5, value=0.05, step=0.01, label="Порог маскирования"
                )
                apply_threshold = gr.Slider(
                    0.5, 1.0, value=0.95, step=0.01, label="Порог применения"
                )
                max_edits = gr.Slider(
                    1, 10, value=1, step=1, label="Максимум правок"
                )

            btn = gr.Button("Распознать", variant="primary")

        with gr.Column():
            output_image = gr.Image(label="Визуализация", type="pil")
            stats_display = gr.Textbox(label="Статистика", interactive=False)

    with gr.Row():
        with gr.Column():
            text_before = gr.Textbox(label="Текст без корректора", lines=10)
            btn_save_recognition = gr.Button("Сохранить в JSON")
            file_recognition = gr.File(label="Результат распознавания")

        with gr.Column():
            text_after = gr.HTML(label="Текст с корректором")
            btn_save_correction = gr.Button("Сохранить в JSON")
            file_correction = gr.File(label="Результат коррекции")

    detector_selector.change(
        update_detector_controls,
        inputs=[detector_selector],
        outputs=[target_size, score_thresh, expand_ratio_w, expand_ratio_h],
    )

    btn.click(
        process_image,
        inputs=[
            input_image,
            detector_selector,
            recognizer_selector,
            corrector_selector,
            target_size,
            score_thresh,
            expand_ratio_w,
            expand_ratio_h,
            mask_threshold,
            apply_threshold,
            max_edits,
        ],
        outputs=[output_image, text_before, text_after, stats_display],
    )

    btn_save_recognition.click(
        save_recognition_json,
        inputs=[],
        outputs=[file_recognition],
    )

    btn_save_correction.click(
        save_correction_json,
        inputs=[],
        outputs=[file_correction],
    )

if __name__ == "__main__":
    demo.launch(share=True)

c:\Users\USER\manuscript-ocr-3\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\USER\AppData\Local\Temp\ipykernel_9600\3043047917.py:254: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="OCR Pipeline", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e3054736e69bf5d699.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[YOLO] Device configuration:
  Requested device: cuda
  Requested providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Running on: CUDAExecutionProvider
[PPOCRv5Rec] Device configuration:
  Requested device: cuda
  Requested providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Running on: CUDAExecutionProvider
[CharLM] Device configuration:
  Requested device: cuda
  Requested providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Running on: CUDAExecutionProvider
[YOLO] Device configuration:
  Requested device: cuda
  Requested providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Running on: CUDAExecutionProvider
[PPOCRv5Rec] Device configuration:
  Requested device: cuda
